In [0]:
import logging
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("main")

In [0]:
%run ./../../common/utilities

In [0]:
dbutils.widgets.text("catalog", "abcgroup", "Catalog")
dbutils.widgets.text("table", "crm_cust_info", "Table")

In [0]:
catalog = dbutils.widgets.get("catalog")
table = dbutils.widgets.get("table")

In [0]:
df = (
    spark.table(f"{catalog}.{bronze_schema}.{table}")
)
display(df.limit(5))


In [0]:
df = df.select(
    [
        F.trim(F.col(field.name)).alias(field.name)
        if isinstance(field.dataType, StringType)
        else F.col(field.name)
        for field in df.schema.fields
    ]
)

In [0]:
df = (
    df
    .withColumn(
        "cst_firstname",
        F.initcap(F.col("cst_firstname"))
    )
    .withColumn(
        "cst_lastname",
        F.initcap(F.col("cst_lastname"))
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("other")
    )
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("other")
    )
)

In [0]:
window = Window.partitionBy("cst_id").orderBy(F.col("cst_create_date").desc_nulls_last())

df_enriched = df.withColumn("_row_num", F.row_number().over(window))
agg = df_enriched.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("_row_num") == 1, 1).otherwise(0)).alias("rows_after_dedup")
).collect()[0]

logger.info("Rows before duplicates dropped: %d", agg["total_rows"])
logger.info("Rows after duplicates dropped: %d", agg["rows_after_dedup"])
df = df_enriched.filter(F.col("_row_num") == 1).drop("_row_num")

In [0]:
COL_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in COL_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
display(df.limit(5))

In [0]:
df_flagged = (
    df.withColumn(
        "is_valid",
        F.when(
            F.col("customer_id").isNotNull() &
            F.col("first_name").isNotNull() &
            F.col("last_name").isNotNull(),
            1
        ).otherwise(0)
    )
)

In [0]:
valid_df = (
    df_flagged
    .filter(F.col("is_valid") == 1)
    .drop("is_valid")
)

invalid_df = (
    df_flagged
    .filter(F.col("is_valid") == 0)
    .drop("is_valid")
)

In [0]:
invalid_df = (
    invalid_df
    .withColumn("error_reason", F.lit("NULL in critical columns"))
    .withColumn("ingestion_ts", F.current_timestamp())
)

In [0]:
invalid_count = invalid_df.count()
if invalid_count > 0:
    logger.warning(f"{invalid_count} invalid records moved to quarantine")

    (
        invalid_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{catalog}.{silver_schema}.{table}_quarantine")
    )

In [0]:
display(valid_df.limit(5))

In [0]:
target_table = f"{catalog}.{silver_schema}.crm_customers"
valid_df.write.mode("overwrite").format("delta").saveAsTable(target_table)

logger.info("Saved table into %s in Delta format.", target_table)

In [0]:
display(valid_df.limit(5))

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {catalog}.{silver_schema}.crm_customers
    LIMIT 5
"""))